# Text2Event–mT5 adaptation — BKEE
Flow: restore pinned full-state → full DEV4039 selection → continue original30-full LR schedule by epoch with patience3/min_delta0.001/grace5 → select best DEV checkpoint → verify DEV and evaluate TEST3728 after early stop or epoch30. Compare pilot best full2 and latest full3 on full DEV before training full4. Substructure handoff was inherited from DEV32 epoch4.


In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.14.1', 'sentencepiece==0.2.1'], check=True)
# Keep Kaggle's CUDA-enabled torch installation; record it in smoke-report.json.


In [ ]:
import sys, subprocess, json, hashlib, zipfile, shutil, importlib.util
from pathlib import Path
working=Path('/kaggle/working'); ROOT=working/'bkee-full-runtime'
def digest(path):
    value=hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda:handle.read(8*1024*1024),b''):value.update(block)
    return value.hexdigest()


In [ ]:
archives=[p for p in Path('/kaggle/input').rglob('bkee-full-runtime.zip') if digest(p)=='9a716433ff2b428ced45192b21cabf6d4fb376e8eac3740b4f9fd73b084ff263']
if len(archives)==1:
    ROOT.mkdir()
    with zipfile.ZipFile(archives[0]) as archive:
        for name in archive.namelist():assert (ROOT/name).resolve().is_relative_to(ROOT.resolve())
        archive.extractall(ROOT)
else:
    manifests=[p for p in Path('/kaggle/input').rglob('full-manifest.json') if digest(p)=='5213e193347550a452575d43d1ee8610099f2a0e7ab514d41282b94ec6624155']
    assert len(manifests)==1,'Expected pinned BKEE runtime'
    shutil.copytree(manifests[0].parent,ROOT)
assert digest(ROOT/'full-manifest.json')=='5213e193347550a452575d43d1ee8610099f2a0e7ab514d41282b94ec6624155'
for name,expected in json.loads((ROOT/'full-manifest.json').read_text()).items():assert digest(ROOT/name)==expected,'Runtime hash differs'
import torch
assert torch.cuda.is_available(),'CUDA required'
print('GPU_PREFLIGHT',torch.__version__,torch.cuda.device_count(),flush=True)


In [ ]:
spec=importlib.util.spec_from_file_location('download',ROOT/'runner/download_text2event_artifact.py')
module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)
metadata=json.loads((ROOT/'resume/metadata.json').read_text());downloads={};paths={}
assert metadata['early_stopping_policy']['enabled'] is True, 'Early stopping must be enabled'
for name,descriptor in metadata['downloads'].items():
    path=working/('source-'+name+'.pt');print('PINNED_DOWNLOAD_START',name,descriptor['expected_bytes'],flush=True)
    downloads[name]=module.download_checked(descriptor['url'],path,descriptor['expected_bytes']);paths[name]=path
(working/'source-download.json').write_text(json.dumps(downloads,indent=2))
output=working/'BKEE-full-run'
command=[sys.executable,'-u',str(ROOT/'runner/run_text2event_bkee_full.py'),
         '--source',str(ROOT/'source'),'--lock',str(ROOT/'locks/vhe-runtime-lock.json'),
         '--output',str(output),'--metadata',str(ROOT/'resume/metadata.json'),
         '--previous-run',str(ROOT/'resume/run.json'),'--commit',str(ROOT/'resume/commit.json'),
         '--checkpoint',str(paths['checkpoint'])]
if 'best' in paths:command+=['--best-checkpoint',str(paths['best'])]
(working/'run-request.json').write_text(json.dumps({'source_version':metadata['source_version'],'source_epoch':metadata['source_epoch'],
 'mode':metadata['mode'],'train_samples':9341,'dev_samples':4039,'full_schedule_epochs':30,
 'test_only_after_training_stop':True,'early_stopping_policy':metadata['early_stopping_policy'],'command':command},indent=2))
with (working/'BKEE-full.log').open('w') as log:
    process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout:print(line,end='',flush=True);log.write(line);log.flush()
    result=process.wait()
assert (output/'status.json').exists(),'Worker failed before status'
status=json.loads((output/'status.json').read_text());(working/'status.json').write_text(json.dumps(status,indent=2))
print('BKEE_FULL_STATUS',status,flush=True)
assert result==0,'Worker failed; inspect pilot.log'
